In [ ]:
import math
from collections import defaultdict


# DATOS DE ENTRENAMIENTO
train_sentences = [
    "I love natural language processing.",
    "I love deep learning.",
    "Deep learning loves big data.",
    "Natural language processing is fun.",
    "Language models are powerful.",
    "I enjoy machine learning.",
    "Machine learning drives AI.",
    "AI is the future.",
    "The future is exciting.",
    "Big data powers AI."
]

test_likelihood = [
    "I love AI.",
    "Machine learning is powerful.",
    "Deep learning powers the future."
]

# Palabras iniciales para la generación solicitada
gen_starts = ["Big", "Machine", "The"]


# PREPROCESAMIENTO
def preprocess(sentence):
    s = sentence.lower().replace(".", "")
    tokens = ["<START>"] + s.split() + ["stop"]
    return tokens

train_tokens = [preprocess(s) for s in train_sentences]
test_tokens = [preprocess(s) for s in test_likelihood]


# UNIGRAMAS Y BIGRAMAS
unigrams = defaultdict(int)
bigrams = defaultdict(int)
vocab = set()

for sentence in train_tokens:
    for i in range(len(sentence)):
        unigrams[sentence[i]] += 1
        vocab.add(sentence[i])
        if i < len(sentence) - 1:
            bigrams[(sentence[i], sentence[i+1])] += 1

V = len(vocab)


# PROBABILIDADES BIGRAMA CON LAPLACE
def bigram_prob(w1, w2):
    return (bigrams[(w1, w2)] + 1) / (unigrams[w1] + V)


# LIKELIHOOD
def sentence_likelihood(tokens):
    prob = 1
    for i in range(len(tokens)-1):
        prob *= bigram_prob(tokens[i], tokens[i+1])
    return prob

print("\n LIKELIHOODS ")
for s, tok in zip(test_likelihood, test_tokens):
    L = sentence_likelihood(tok)
    print(s, "=>", L)


# GENERACIÓN
def generate(start_word, max_len=20):
    word = start_word.lower()
    sentence = [word]

    for _ in range(max_len):

        best_next = None
        best_p = -1

        for w in vocab:
            if w == "<START>":
                continue
            p = bigram_prob(word, w)
            if p > best_p:
                best_p = p
                best_next = w

        if best_next == "stop":
            sentence.append("STOP")
            break

        sentence.append(best_next)
        word = best_next

    return " ".join(sentence)

print("\n GENERACIÓN ")
for s in gen_starts:
    print(s, "=>", generate(s))


# PERPLEXITY
def perplexity(tokens):
    N = len(tokens)
    log_sum = 0
    for i in range(len(tokens)-1):
        p = bigram_prob(tokens[i], tokens[i+1])
        log_sum += math.log(p)
    return math.exp(-(log_sum / N))

print("\n PERPLEXITY TEST ")
P_test = []
for s, tok in zip(test_likelihood, test_tokens):
    PP = perplexity(tok)
    P_test.append(PP)
    print(s, "=>", PP)

print("Perplexity promedio test:", sum(P_test)/len(P_test))

print("\n PERPLEXITY TRAIN ")
P_train = []
for tok in train_tokens:
    PP = perplexity(tok)
    P_train.append(PP)
    print(" ".join(tok), "=>", PP)

print("Perplexity promedio train:", sum(P_train)/len(P_train))



 LIKELIHOODS 
I love AI. => 4.8590864917395524e-05
Machine learning is powerful. => 6.014784339907492e-07
Deep learning powers the future. => 6.930615700304929e-08

 GENERACIÓN 
Big => big data STOP
Machine => machine learning STOP
The => the future STOP

 PERPLEXITY TEST 
I love AI. => 7.289354541774024
Machine learning is powerful. => 10.884203587782363
Deep learning powers the future. => 10.537725543689364
Perplexity promedio test: 9.570427891081918

 PERPLEXITY TRAIN 
<START> i love natural language processing stop => 7.430170977413303
<START> i love deep learning stop => 6.7657556433587
<START> deep learning loves big data stop => 8.64446883539319
<START> natural language processing is fun stop => 8.646045142830289
<START> language models are powerful stop => 9.019158591231648
<START> i enjoy machine learning stop => 7.193381863475782
<START> machine learning drives ai stop => 8.074298140748521
<START> ai is the future stop => 8.642612995749106
<START> the future is exciting stop